In [4]:
import time
import numpy as np

from data_generation import generate_data
from practical_transmission import practical_transmission

In [5]:
H = [5, 10, 15, 20]
RHO = [0.1, 0.2, 0.4]
n_rep = 2

In [6]:
def run(h, rho, n_rep, n_test=2000):
    est_trans = np.zeros(n_rep)
    est_single = np.zeros(n_rep)

    pred_trans = np.zeros(n_rep)
    pred_single = np.zeros(n_rep)

    for i in range(n_rep):
        d = generate_data(h=h, rho=rho, seed=i)
        res = practical_transmission(d["X0"], d["y0"], d["X"], d["y"])

        # squared estimation error của coefficient
        est_trans[i] = ((res["coef"] - d["beta"]) ** 2).sum()
        est_single[i] = ((res["beta_T_cv"] - d["beta"]) ** 2).sum()

        p = d["X0"].shape[1]
        rng = np.random.default_rng(10_000 + i)

        # cùng phân phối train
        X_test = rng.standard_normal((n_test, p))
        y_test = X_test @ d["beta"] + rng.standard_normal(n_test)

        # prediction loss
        pred_trans[i] = np.mean((y_test - X_test @ res["coef"]) ** 2)
        pred_single[i] = np.mean((y_test - X_test @ res["beta_T_cv"]) ** 2)

    return (est_trans.mean(), est_single.mean(), pred_trans.mean(), pred_single.mean())

In [7]:
# est  = ||beta_hat - beta||^2   
# pred = MSE tren tap test target   (~ sigma^2 + est, voi sigma^2 = 1)

print(f"{'h'} {'rho'} | {'est_trans'} {'est_single'} | {'pred_trans'} {'pred_single'}")
print("-" * 48)

for h in H:
    for rho in RHO:
        est_trans, est_single, pred_trans, pred_single = run(h, rho, n_rep)

        print(f"{h} {rho} || {est_trans} {est_single}" f"|| {pred_trans} {pred_single}")

h rho | est_trans est_single | pred_trans pred_single
------------------------------------------------
5 0.1 || 0.23190300712867834 0.5929676238047248|| 1.23696276561095 1.6260712699312723
5 0.2 || 0.20919565234188853 0.5929676238047248|| 1.2174731552681965 1.6260712699312723
5 0.4 || 0.17974326688832054 0.5929676238047248|| 1.1753144833583347 1.6260712699312723
10 0.1 || 0.3487976011439823 0.5929676238047248|| 1.3652124691039291 1.6260712699312723
10 0.2 || 0.36302250815357473 0.5929676238047248|| 1.3728600459892268 1.6260712699312723
10 0.4 || 0.3306353713849371 0.5929676238047248|| 1.3375424440054808 1.6260712699312723
15 0.1 || 0.4327461964351955 0.5929676238047248|| 1.4425922980648438 1.6260712699312723
15 0.2 || 0.49225169677338715 0.5929676238047248|| 1.5092330313738431 1.6260712699312723
15 0.4 || 0.46210305223529724 0.5929676238047248|| 1.465360745861854 1.6260712699312723
20 0.1 || 0.5101497168109219 0.5929676238047248|| 1.5102652389458067 1.6260712699312723
20 0.2 || 0.56221

# Feature Selection

In [14]:
h, rho, seed = 10, 0.2, 0

d = generate_data(h=h, rho=rho, seed=seed)
res = practical_transmission(d["X0"], d["y0"], d["X"], d["y"])

bh = res["coef"]
bt = d["beta"]
nt, p = d["X0"].shape




In [16]:
print(res["lam0_hat"])
print(res["lam0_grid"])

0.024414231059462888
[2.55767331 2.41319571 2.27687936 2.14826323 2.02691235 1.91241633
 1.80438795 1.70246187 1.60629338 1.51555724 1.4299466  1.34917192
 1.27296003 1.20105319 1.13320822 1.06919567 1.00879905 0.95181412
 0.89804814 0.84731928 0.799456   0.75429641 0.71168779 0.67148604
 0.63355521 0.597767   0.5640004  0.53214121 0.50208167 0.47372013
 0.44696067 0.4217128  0.39789113 0.3754151  0.35420869 0.33420019
 0.31532192 0.29751005 0.28070433 0.26484793 0.24988723 0.23577162
 0.22245338 0.20988746 0.19803136 0.18684498 0.1762905  0.16633222
 0.15693646 0.14807145 0.13970721 0.13181544 0.12436946 0.11734409
 0.11071557 0.10446148 0.09856067 0.09299319 0.0877402  0.08278394
 0.07810765 0.07369551 0.06953261 0.06560485 0.06189897 0.05840243
 0.0551034  0.05199073 0.04905388 0.04628293 0.0436685  0.04120176
 0.03887436 0.03667843 0.03460654 0.03265169 0.03080727 0.02906703
 0.02742509 0.02587591 0.02441423 0.02303512 0.02173392 0.02050621
 0.01934786 0.01825494 0.01722376 0.01625

In [21]:
coef = res["coef"]

print("Số beta = 0:", np.sum(coef == 0))
print("Số beta != 0:", np.sum(coef != 0))
print("Tổng số beta:", len(coef))

Số beta = 0: 393
Số beta != 0: 107
Tổng số beta: 500
